## Update strategies — `RollingUpdate` vs `Recreate`

`spec.strategy.type` picks between two algorithms.

**`RollingUpdate`** (default) — the new ReplicaSet scales up while the old scales down, governed by `maxSurge` / `maxUnavailable`. **No downtime** if your app handles `SIGTERM` cleanly and your readiness probe is honest.

**`Recreate`** — the old ReplicaSet is scaled to `0` *first*, then the new one scales up. **There is a window with zero pods.** Use it when:

- The new version can't coexist with the old (incompatible DB schema migration, conflicting filesystem state).
- A `ReadWriteOnce` PersistentVolume can be mounted by only one pod at a time (notebook 06).
- A `HostPort` means two pods on a node would briefly fight for it.

`Recreate` is simpler; `RollingUpdate` is the production default. The CKA wants both names and the *kind* of situation that argues for `Recreate`.

### What triggers a rollout

**Fires** whenever `spec.template` changes — image, env, command, args, mounts, resources, probes, security context, adding/removing a container or volume.

**Does not fire** for: `spec.replicas` (a scale), Deployment-level fields outside the template (`strategy`, `revisionHistoryLimit`), or labels on the Deployment itself.

To force a rollout *without* a real template change — e.g. to pick up a new ConfigMap (notebook 05) — bump a template annotation:

```bash
kubectl patch deployment web -p \
  '{"spec":{"template":{"metadata":{"annotations":{"rolled-at":"'$(date +%s)'"}}}}}'
```

On our map this is the **Rollout** box holding both chips — **rolling** (the default) and **recreate** (the coexistence-forbidden case).